In [ ]:
%%sql
DROP TABLE IF EXISTS silver_sone_srpatient;

CREATE TABLE silver_sone_srpatient AS (
    SELECT
        a.RowIdentifier AS id,
        DateBirth AS date_birth,
        Gender AS gender,
        TestPatient AS test_patient,
        RemovedData AS removed_data,
        to_date(reverse(substring(reverse(FILEDATE), 5, 8)), "yyyyMMdd") AS file_date,
        CASE
            WHEN LEFT(FILEDATE, 5) = 'Y0600' THEN LEFT(FILEDATE, 6)
            WHEN LEFT(FILEDATE, 5) NOT IN ('G4B9E', 'O0D1Z', 'T7L2F', 'U1V5S')
                 AND to_date(reverse(substring(reverse(FILEDATE), 5, 8)), "yyyyMMdd") < '2023-12-01'
            THEN 'OOD1Z'
            ELSE LEFT(FILEDATE, 5)
        END AS id_organisation_source
    FROM bronze_sone_srpatient a
    INNER JOIN (
        SELECT
            RowIdentifier,
            CASE
                WHEN LEFT(FILEDATE, 5) = 'Y0600' THEN LEFT(FILEDATE, 6)
                WHEN LEFT(FILEDATE, 5) NOT IN ('G4B9E', 'OOD1Z', 'T7L2F', 'U1V5S')
                     AND to_date(reverse(substring(reverse(FILEDATE), 5, 8)), "yyyyMMdd") < '2023-12-01'
                THEN 'O0D1Z'
                ELSE LEFT(FILEDATE, 5)
            END AS id_organisation_source,
            reverse(substring(reverse(FILEDATE), 5, 8)) AS cleaned_filedate,
            ROW_NUMBER() OVER (
                PARTITION BY RowIdentifier
                ORDER BY to_date(reverse(substring(reverse(FILEDATE), 5, 8)), "yyyyMMdd") DESC
            ) AS RowNo
        FROM bronze_sone_srpatient
    ) b
        ON a.RowIdentifier = b.RowIdentifier
       AND b.RowNo = 1
       AND reverse(substring(reverse(a.FILEDATE), 5, 8)) = b.cleaned_filedate
       AND b.id_organisation_source = CASE
           WHEN LEFT(a.FILEDATE, 5) = 'Y0600' THEN LEFT(a.FILEDATE, 6)
           WHEN LEFT(a.FILEDATE, 5) NOT IN ('G4B9E', 'OOD1Z', 'T7L2F', 'U1V5S')
                AND to_date(reverse(substring(reverse(a.FILEDATE), 5, 8)), "yyyyMMdd") < '2023-12-01'
           THEN 'OOD1Z'
           ELSE LEFT(a.FILEDATE, 5)
       END
);

In [ ]:
MPB: Completed MPB mapping for CARE EPISODE.care_epi_is_test. Source confirmed from silver_drj_tenancies using tenancy name logic as per approved Monday definition. Implemented 1/0 flag in Care Episode notebook and validated output successfully.

S1 / SystemOne task note;

Completed SystemOne mapping for CARE EPISODE.care_epi_is_test. Identified that source field TestPatient existed in bronze_sone_srpatient and first added it into silver_sone_srpatient, then used the patient-level flag in the Care Episode notebook to derive the 1/0 output. Mapping and validation completed successfully.